In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/audiocraft_project

In [20]:
!git clone -b master https://github.com/Kikongo/audiocraft.git
%cd audiocraft

/content/audiocraft


In [19]:
!pip install -e .

Obtaining file:///content/audiocraft/audiocraft
ERROR: file:///content/audiocraft/audiocraft does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [4]:
!python create_manifests.py --data_dir musiccaps_audio --output_dir dataset/musiccaps_audio

2026-03-17 12:11:39.390052: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773749499.414588    4044 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773749499.422483    4044 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773749499.441247    4044 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773749499.441307    4044 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773749499.441316    4044 computation_placer.cc:177] computation placer alr

In [ ]:
from audiocraft.models import MusicGen

model = MusicGen.get_pretrained('facebook/musicgen-small')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


state_dict.bin:   0%|          | 0.00/841M [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

compression_state_dict.bin:   0%|          | 0.00/236M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [ ]:
!dora run solver=musicgen/musicgen_finetune_final

2026-03-17 16:27:08.409056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773764828.435985   70236 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773764828.443028   70236 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773764828.461439   70236 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773764828.461489   70236 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773764828.461493   70236 computation_placer.cc:177] computation placer alr

In [46]:
import torch
from audiocraft.models import MusicGen

model = MusicGen.get_pretrained('debug', device='cuda')

checkpoint_path = "/tmp/audiocraft/xps/819cf66b/checkpoint.th"

# Load the state dictionary from the checkpoint file onto the correct device
state_dict = torch.load(checkpoint_path, map_location=model.device, weights_only=False)

# Load the state dictionary into the model
model.lm.load_state_dict(state_dict['model'])

audio = model.generate(["energetic rock with electric guitar"])

model.save_audio(audio, "result_debug_model.wav")
print("Generated audio saved to result_debug_model.wav")

RuntimeError: Error(s) in loading state_dict for LMModel:
	Missing key(s) in state_dict: "condition_provider.conditioners.description.embed.weight", "transformer.layers.0.self_attn.in_proj_bias", "transformer.layers.0.self_attn.out_proj.bias", "transformer.layers.0.linear1.bias", "transformer.layers.0.linear2.bias", "transformer.layers.0.cross_attention.in_proj_bias", "transformer.layers.0.cross_attention.out_proj.bias", "transformer.layers.1.self_attn.in_proj_bias", "transformer.layers.1.self_attn.out_proj.bias", "transformer.layers.1.linear1.bias", "transformer.layers.1.linear2.bias", "transformer.layers.1.cross_attention.in_proj_bias", "transformer.layers.1.cross_attention.out_proj.bias", "linears.0.bias", "linears.1.bias", "linears.2.bias", "linears.3.bias". 
	Unexpected key(s) in state_dict: "out_norm.weight", "out_norm.bias". 
	size mismatch for condition_provider.conditioners.description.output_proj.weight: copying a param with shape torch.Size([64, 512]) from checkpoint, the sh

In [50]:
# Exporting models
from audiocraft.utils import export
from audiocraft import train
export.export_lm('/tmp/audiocraft/xps/819cf66b/checkpoint.th', '/checkpoints/my_audio_lm/state_dict.bin')
export.export_pretrained_compression_model('/debug_compression_model', '/checkpoints/my_audio_lm/compression_state_dict.bin')

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL omegaconf.dictconfig.DictConfig was not an allowed global by default. Please use `torch.serialization.add_safe_globals([omegaconf.dictconfig.DictConfig])` or the `torch.serialization.safe_globals([omegaconf.dictconfig.DictConfig])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to 

In [14]:
!dora grid musicgen.musicgen_style_32khz --dry_run --init

2026-03-17 14:30:03.132805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773757803.154491   40005 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773757803.161644   40005 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773757803.179903   40005 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773757803.179949   40005 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773757803.179954   40005 computation_placer.cc:177] computation placer alr

In [ ]:
import torch
import gc


# Run garbage collection
gc.collect()

# Clear the PyTorch CUDA cache
torch.cuda.empty_cache()

In [ ]:
!hf auth login

A new version of huggingface_hub (1.7.1) is available! You are using version 1.6.0.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: y
Token is valid (permission: fineGrained).
The token `test` has been saved to /root/.cache/huggingface/st